# 🧪 SMILES → SDF Dönüştürücü (Sanal Tarama Hazırlığı)

Bu notebook, **kendi moleküllerinizi** bir CSV dosyasından okuyup docking / sanal tarama
(ör. PyRx, AutoDock Vina) için **3B SDF** formatına dönüştürür.

**Ne yapar?**
1. Google Drive'a bağlanır ve **sizin belirttiğiniz isimde** bir çalışma klasörü oluşturur.
2. O klasöre koyduğunuz **CSV dosyasındaki SMILES** notasyonlarını okur.
3. Her molekülü temizler (tuz/karışım ayıklar), 3B koordinat üretir (ETKDG) ve MMFF94 ile enerji minimizasyonu yapar.
4. Sonucu **iki biçimde** kaydeder:
   - `tum_molekuller.sdf` → hepsi **tek** SDF dosyasında (PyRx'e toplu yükleme için)
   - `sdf_files/` klasörü → **her molekül için ayrı** SDF dosyası

> ℹ️ Hücreleri **yukarıdan aşağıya sırayla** çalıştırın. Sarı `# 👉 BURAYI DEĞİŞTİRİN` ile
> işaretli satırları kendi çalışmanıza göre düzenleyin.


## 1️⃣ Google Drive'a bağlan ve çalışma klasörünü oluştur

Aşağıdaki hücrede **`KLASOR_ADI`** değişkenini kendi çalışmanıza göre yazın
(BÜYÜK harflerle bırakılan yazıyı silip kendi klasör adınızı yazın).
Bu klasör Drive'ınızda `MyDrive` içinde oluşturulacak; tüm çıktılar buraya kaydedilecek.


In [ ]:
from google.colab import drive
import os

# Drive'i bagla (ilk seferde izin/onay isteyecek)
drive.mount('/content/drive')

# ==========================================================
# 👉 BURAYI DEĞİŞTİRİN — kendi klasör adınızı yazın
KLASOR_ADI = "BURAYA_KLASOR_ADINIZI_YAZIN"
# ==========================================================

ANA_YOL = f"/content/drive/MyDrive/{KLASOR_ADI}"
os.makedirs(ANA_YOL, exist_ok=True)

print("✅ Çalışma klasörü hazır:")
print("   ", ANA_YOL)


## 2️⃣ CSV dosyanızı çalışma klasörüne yükleyin

CSV dosyanız en az bir **SMILES** kolonu içermelidir (isteğe bağlı olarak molekül **isim** kolonu da olabilir).

İki seçeneğiniz var:

- **A) Buradan yükleyin:** Aşağıdaki hücreyi çalıştırın, açılan pencereden CSV dosyanızı seçin.
  Dosya otomatik olarak çalışma klasörünüze kopyalanır.
- **B) Elle koydunuzsa:** Dosyayı Drive'da çalışma klasörünüze zaten koyduysanız, bu hücreyi atlayın.

Ardından **`CSV_DOSYA_ADI`** değişkenine kendi dosya adınızı yazın.


In [ ]:
from google.colab import files
import shutil

# --- Seçenek A: buradan yükle (dosya secme penceresi acilir) ---
# Elle koyduysaniz bu satiri yorum yapabilirsiniz (basina # koyun).
yuklenen = files.upload()

for fn in yuklenen:
    hedef = os.path.join(ANA_YOL, fn)
    shutil.move(fn, hedef)
    print("📁 Klasöre kaydedildi:", hedef)


In [ ]:
# ==========================================================
# 👉 BURAYI DEĞİŞTİRİN — kendi CSV dosyanızın adını yazın
CSV_DOSYA_ADI = "BURAYA_CSV_DOSYA_ADINIZI_YAZIN.csv"
# ==========================================================

csv_yolu = os.path.join(ANA_YOL, CSV_DOSYA_ADI)
assert os.path.exists(csv_yolu), f"❌ Dosya bulunamadı: {csv_yolu}\n   Klasördeki dosyalar: {os.listdir(ANA_YOL)}"
print("✅ CSV bulundu:", csv_yolu)


## 3️⃣ Gerekli kütüphaneyi kur (RDKit)

Kimyasal işlemler için **RDKit** kütüphanesini kuruyoruz. (Yaklaşık 15–30 saniye sürer.)


In [ ]:
!pip install rdkit -q
import rdkit
print("✅ RDKit sürümü:", rdkit.__version__)


## 4️⃣ CSV'yi oku ve kolonları kontrol et

Dosya okunur ve ilk birkaç satır gösterilir. Notebook, ayırıcıyı (virgül `,` veya noktalı virgül `;`)
otomatik algılamaya çalışır. Aşağıdaki tabloda **SMILES kolonunuzun adını** ve varsa **isim kolonunuzun adını** not edin.


In [ ]:
import pandas as pd

# Ayiriciyi otomatik algila (virgul / noktali virgul / tab)
df = pd.read_csv(csv_yolu, sep=None, engine="python")

print("Satır x Kolon:", df.shape)
print("Kolonlar:", list(df.columns))
df.head()


Aşağıda **`SMILES_KOLONU`** değişkenine yukarıda gördüğünüz SMILES kolonunun adını yazın.
Molekül isimleri için bir kolon varsa **`ISIM_KOLONU`** değişkenine onu da yazın; yoksa `None` bırakın
(bu durumda moleküller `mol_1`, `mol_2`, ... diye adlandırılır).


In [ ]:
# ==========================================================
# 👉 BURAYI DEĞİŞTİRİN — kolon adlarını kendi CSV'nize göre yazın
SMILES_KOLONU = "SMILES"      # SMILES notasyonlarının olduğu kolon
ISIM_KOLONU   = None          # örn. "Name" ya da "Drug_Name"; yoksa None bırakın
# ==========================================================

# Otomatik yardimci: yaygin SMILES kolon adlarini dener
if SMILES_KOLONU not in df.columns:
    for aday in ["SMILES", "Smiles", "smiles", "canonical_smiles", "Canonical Smiles"]:
        if aday in df.columns:
            SMILES_KOLONU = aday
            print(f"ℹ️ SMILES kolonu otomatik seçildi: '{aday}'")
            break

assert SMILES_KOLONU in df.columns, f"❌ '{SMILES_KOLONU}' kolonu yok. Mevcut kolonlar: {list(df.columns)}"
if ISIM_KOLONU is not None:
    assert ISIM_KOLONU in df.columns, f"❌ '{ISIM_KOLONU}' kolonu yok. Mevcut kolonlar: {list(df.columns)}"
print(f"✅ SMILES kolonu: '{SMILES_KOLONU}'  |  İsim kolonu: {ISIM_KOLONU}")
print(f"   Toplam satır: {len(df)}")


## 5️⃣ SMILES → 3B SDF dönüştür ve kaydet

Her molekül için sırasıyla:
1. SMILES okunur, geçersizse atlanır (rapor edilir).
2. Tuz / karışım varsa **en büyük organik fragman** alınır.
3. Hidrojen eklenir, **ETKDG** ile 3B koordinat üretilir, **MMFF94** ile enerji minimizasyonu yapılır.
4. Molekül hem **tek toplu SDF**'e hem de **`sdf_files/`** içindeki **ayrı** dosyasına yazılır.

Çıktılar:
- `tum_molekuller.sdf` — hepsi tek dosyada (PyRx'e toplu import için ideal)
- `sdf_files/<molekül_adı>.sdf` — her molekül ayrı dosyada


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import RDLogger
import re
RDLogger.DisableLog("rdApp.*")

# Cikti yollari
sdf_files_dir = os.path.join(ANA_YOL, "sdf_files")
os.makedirs(sdf_files_dir, exist_ok=True)
tek_sdf_yolu = os.path.join(ANA_YOL, "tum_molekuller.sdf")

def dosya_adi_temizle(ad):
    """Dosya adinda gecersiz karakterleri temizler."""
    ad = re.sub(r"[^A-Za-z0-9._-]+", "_", str(ad)).strip("_")
    return ad or "mol"

def en_buyuk_fragman(mol):
    try:
        return rdMolStandardize.FragmentParent(mol)
    except Exception:
        frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=True)
        return max(frags, key=lambda m: m.GetNumHeavyAtoms()) if frags else mol

writer = Chem.SDWriter(tek_sdf_yolu)
basarili, basarisiz = 0, []
kullanilan_adlar = {}

for i, satir in df.iterrows():
    smi = str(satir[SMILES_KOLONU]).strip()
    if ISIM_KOLONU is not None and pd.notna(satir[ISIM_KOLONU]) and str(satir[ISIM_KOLONU]).strip():
        ham_ad = str(satir[ISIM_KOLONU]).strip()
    else:
        ham_ad = f"mol_{i+1}"

    if not smi or smi.lower() in ("nan", "none", ""):
        basarisiz.append((ham_ad, "boş SMILES")); continue

    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        basarisiz.append((ham_ad, "geçersiz SMILES")); continue

    mol = en_buyuk_fragman(mol)
    if mol is None or mol.GetNumHeavyAtoms() == 0:
        basarisiz.append((ham_ad, "boş molekül")); continue

    # 3B koordinat + minimizasyon
    molH = Chem.AddHs(mol)
    p = AllChem.ETKDGv3(); p.randomSeed = 42
    if AllChem.EmbedMolecule(molH, p) != 0:
        if AllChem.EmbedMolecule(molH, useRandomCoords=True, randomSeed=42) != 0:
            basarisiz.append((ham_ad, "3B üretilemedi")); continue
    try:
        AllChem.MMFFOptimizeMolecule(molH, maxIters=500)
    except Exception:
        pass

    # Benzersiz isim (ayni ad tekrar ederse _2, _3 ekle)
    temiz = dosya_adi_temizle(ham_ad)
    kullanilan_adlar[temiz] = kullanilan_adlar.get(temiz, 0) + 1
    if kullanilan_adlar[temiz] > 1:
        temiz = f"{temiz}_{kullanilan_adlar[temiz]}"

    molH.SetProp("_Name", ham_ad)
    molH.SetProp("SMILES", smi)

    # 1) toplu SDF'e yaz
    writer.write(molH)

    # 2) ayri SDF'e yaz
    tekil = Chem.SDWriter(os.path.join(sdf_files_dir, f"{temiz}.sdf"))
    tekil.write(molH)
    tekil.close()

    basarili += 1

writer.close()

print(f"✅ Başarılı: {basarili} molekül")
print(f"❌ Başarısız: {len(basarisiz)} molekül")
if basarisiz:
    print("\nBaşarısız olanlar:")
    for ad, neden in basarisiz[:50]:
        print(f"   - {ad}: {neden}")
    if len(basarisiz) > 50:
        print(f"   ... ve {len(basarisiz)-50} tane daha")


## 6️⃣ Özet ve doğrulama

Aşağıdaki hücre çıktıların yerini ve içeriğini özetler. İşiniz bittiğinde tüm dosyalar
Drive'daki çalışma klasörünüzde hazır olacak.


In [ ]:
# Toplu SDF'i tekrar okuyup dogrula
supp = Chem.SDMolSupplier(tek_sdf_yolu, removeHs=False)
okunan = [m for m in supp if m is not None]
tekil_sayi = len([f for f in os.listdir(sdf_files_dir) if f.endswith(".sdf")])

print("=" * 55)
print("📦 ÇIKTILAR")
print("=" * 55)
print(f"📁 Çalışma klasörü : {ANA_YOL}")
print(f"📄 Toplu SDF       : {tek_sdf_yolu}")
print(f"   → içindeki molekül sayısı: {len(okunan)}")
print(f"📁 Ayrı SDF klasörü: {sdf_files_dir}")
print(f"   → dosya sayısı           : {tekil_sayi}")
print("=" * 55)
print("\n➡️  'tum_molekuller.sdf' dosyasını PyRx'e Open Babel üzerinden yükleyebilirsiniz.")
